In [ ]:
"""
GoEmotions Concept-Frustration Pipeline (RoBERTa sentiment concepts + DeBERTa embeddings)
======================================================================================

MINIMAL EDITS IN THIS VERSION (annotated with ### NEW ###):
  1) Standardize DeBERTa embeddings per fold (fit on train, apply to test).
  2) Normalize concept logits per fold for CBM training (fit on train, apply to test),
     BUT keep the original raw C for geometry/cov metrics + ridge-on-C3 so results remain comparable.
  3) Add F1 reporting for BB + CBMs (threshold=0.5).
  4) Add CBM2 pair-specific frustration metrics for the (C1,C2) pair (both Fisher and Euclid).
  5) ### NEW ### Replace GoEmotions loader with local Sarcasm Headlines JSONL loader.
     - Task label Y is now df["is_sarcastic"] (0/1).
     - Texts come from df["headline"].
     - Everything else (concepts=C from RoBERTa sentiment, X from DeBERTa embeddings, training) unchanged.
"""

import os
import time
import hashlib
import json  # ### NEW ###
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

# from datasets import load_dataset  # ### NEW ### no longer needed (kept commented to be explicit)
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification


# ============================================================
# Config
# ============================================================

SPLIT = "train"   # kept for filename compatibility; not used for local JSONL
N_DATA = 10000
SEED_DATA = 123

# ---------------------------
# ### NEW ### Local sarcasm dataset config
# ---------------------------
DATA_DIR = "headlines_data"
FILES = [
    "Sarcasm_Headlines_Dataset.json",
    "Sarcasm_Headlines_Dataset_v2.json",
]

def load_jsonl(path: str) -> pd.DataFrame:
    """Load a JSON-lines file where each line is a JSON object."""
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

# ---------------------------
# Choose task here (kept for compatibility; no longer used for Y)
# ---------------------------
TASK_MODE = "neu"      # unused for sarcasm Y; kept for filenames
TASK_LOGIC = "any"     # unused for sarcasm Y; kept for filenames

POS_LABELS = [
    "admiration","amusement","approval","caring","desire","excitement","gratitude",
    "joy","love","optimism","pride","relief"
]
NEG_LABELS = [
    "anger","annoyance","disappointment","disapproval","disgust","embarrassment",
    "fear","grief","nervousness","remorse","sadness"
]
NEU_LABELS = ["neutral","confusion","curiosity","realization","surprise"]

K_FOLDS = 10
SEED_FOLDS = 777

BATCH_SIZE_SENT = 64
BATCH_SIZE_EMB  = 64
MAX_LEN = 128

SENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
EMB_MODEL  = "microsoft/deberta-v3-base"

OUT_DIR = "goemotions_frustration_runs"
os.makedirs(OUT_DIR, exist_ok=True)

# Training params
P_LO, P_HI = 0.2, 0.8
MIN_KEEP = 500

HIDDEN_BB = 128
EPOCHS_BB = 60

EPOCHS_CBM = 30
LAMBDA_C = 1.0

K_SAE = 300
EPOCHS_SAE = 60

PAIR_SOFT_EPS = 1e-6
RIDGE_LAM_C3 = 1e-3

# ============================================================
# (normalization toggles)
# ============================================================
STANDARDIZE_X_PER_FOLD = True
NORMALIZE_C_FOR_CBM = True
NORM_EPS = 1e-6


# ============================================================
# Device
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# Task hashing -> cache-safe filenames
# ============================================================

def _norm_label(s: str) -> str:
    s = str(s).strip().lower()
    return s[6:] if s.startswith("label_") else s

def _task_signature() -> str:
    payload = {
        # ### NEW ### include dataset identity + local file list
        "DATASET": "sarcasm_headlines_jsonl",
        "DATA_DIR": DATA_DIR,
        "FILES": list(FILES),

        # kept for backwards compatibility in filenames/logs (not used for Y now)
        "TASK_MODE": _norm_label(TASK_MODE),
        "TASK_LOGIC": _norm_label(TASK_LOGIC),

        "SPLIT": SPLIT,
        "N_DATA": int(N_DATA),
        "SEED_DATA": int(SEED_DATA),
        "SENT_MODEL": SENT_MODEL,
        "EMB_MODEL": EMB_MODEL,
        "MAX_LEN": int(MAX_LEN),
    }
    b = repr(payload).encode("utf-8")
    return hashlib.md5(b).hexdigest()[:10]

TASK_SIG = _task_signature()

# ### NEW ### rename cache/results to avoid mixing with GoEmotions cache
CACHE_NPZ = os.path.join(
    OUT_DIR, f"cache_sarcasm_{TASK_SIG}_N{N_DATA}_seed{SEED_DATA}.npz"
)
RESULTS_CSV = os.path.join(
    OUT_DIR, f"results_sarcasm_{TASK_SIG}_N{N_DATA}_K{K_FOLDS}_t{int(time.time())}.csv"
)


# ============================================================
# 1) Models (unchanged)
# ============================================================

class BlackBoxMLP(nn.Module):
    def __init__(self, r: int, hidden: int = 128):
        super().__init__()
        self.fc1 = nn.Linear(r, hidden)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)

    def forward(self, x, return_hidden: bool = False):
        z = self.fc1(x)
        h = self.act(z)
        logit = self.fc2(h).squeeze(-1)
        if return_hidden:
            return logit, z, h
        return logit


class LinearCBM(nn.Module):
    def __init__(self, xdim: int, k_known: int):
        super().__init__()
        self.concept = nn.Linear(xdim, k_known)
        self.task = nn.Linear(k_known, 1)

    def forward(self, x):
        c = self.concept(x)
        y = self.task(c).squeeze(-1)
        return c, y


class SparseAE(nn.Module):
    def __init__(self, xdim: int, K: int):
        super().__init__()
        self.W = nn.Linear(xdim, K, bias=True)
        self.D = nn.Parameter(torch.randn(K, xdim) * 0.02)

    def forward(self, x):
        s = torch.relu(self.W(x))
        x_hat = s @ self.D
        return s, x_hat


# ============================================================
# 2) Helpers
# ============================================================

@torch.no_grad()
def accuracy_from_logits(logits: torch.Tensor, y_true_np: np.ndarray) -> float:
    probs = torch.sigmoid(logits).cpu().numpy()
    preds = (probs >= 0.5).astype(np.int64)
    return float((preds == y_true_np).mean())

@torch.no_grad()
def f1_from_logits(logits: torch.Tensor, y_true_np: np.ndarray, thresh: float = 0.5) -> float:
    y_true = np.asarray(y_true_np, dtype=np.int64).reshape(-1)
    probs = torch.sigmoid(logits).detach().cpu().numpy().reshape(-1)
    y_pred = (probs >= thresh).astype(np.int64)
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    denom = (2 * tp + fp + fn)
    return float((2 * tp) / denom) if denom > 0 else 0.0


# ============================================================
# 3) Fisher helpers (unchanged)
# ============================================================

@torch.no_grad()
def bb_predict_proba(bb: BlackBoxMLP, X: np.ndarray, *, device="cpu") -> np.ndarray:
    bb.eval()
    Xt = torch.tensor(X, dtype=torch.float32, device=device)
    logits = bb(Xt)
    return torch.sigmoid(logits).detach().cpu().numpy()


@torch.no_grad()
def compute_fisher_on_input_x(bb: BlackBoxMLP, X: np.ndarray, *, device="cpu", ridge=1e-6):
    bb.eval()
    Xt = torch.tensor(X, dtype=torch.float32, device=device)

    logits, z, _ = bb(Xt, return_hidden=True)
    p = torch.sigmoid(logits)
    s = p * (1 - p)
    m = (z > 0).float()

    W_H = bb.fc1.weight
    w_l = bb.fc2.weight.squeeze(0)

    U = m * w_l.unsqueeze(0)
    G = U @ W_H

    Fm = (G.T @ (G * s.unsqueeze(1))) / float(X.shape[0])
    Fm = Fm + ridge * torch.eye(Fm.shape[0], device=device)
    return Fm.cpu().numpy()


# ============================================================
# 4) Training (unchanged)
# ============================================================

def train_bb_minibatch(X_tr, y_tr, X_te, y_te, *, hidden=128, epochs=30, batch_size=512, lr=1e-3, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    r = X_tr.shape[1]
    model = BlackBoxMLP(r, hidden=hidden).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)
    Xv = torch.tensor(X_te, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb, yb = Xt[b], yt[b]
            opt.zero_grad()
            loss = bce(model(xb), yb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        acc_te = accuracy_from_logits(model(Xv), y_te)
    return model, acc_te


def train_cbm_linear_minibatch(X_tr, Ck_tr, y_tr, X_te, Ck_te, y_te, *,
                              epochs=30, batch_size=512, lr=1e-3, lambda_c=1.0, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    xdim = X_tr.shape[1]
    k = Ck_tr.shape[1]
    model = LinearCBM(xdim, k).to(device)

    opt = optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    mse = nn.MSELoss()

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)
    Ct = torch.tensor(Ck_tr, dtype=torch.float32, device=device)
    yt = torch.tensor(y_tr, dtype=torch.float32, device=device)

    Xv = torch.tensor(X_te, dtype=torch.float32, device=device)
    Cv = torch.tensor(Ck_te, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb, cb, yb = Xt[b], Ct[b], yt[b]
            opt.zero_grad()
            c_hat, logit = model(xb)
            loss = bce(logit, yb) + lambda_c * mse(c_hat, cb)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        c_hat_te, logit_te = model(Xv)
        acc_te = accuracy_from_logits(logit_te, y_te)
        mse_te = float(((c_hat_te - Cv)**2).mean().cpu().item())

    return model, acc_te, mse_te


def train_sae_minibatch(X_tr, *, K=60, epochs=60, batch_size=512, lr=2e-3, l1=1e-3, seed=0, device="cpu"):
    torch.manual_seed(seed); np.random.seed(seed)
    xdim = X_tr.shape[1]
    model = SparseAE(xdim, K).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)

    Xt = torch.tensor(X_tr, dtype=torch.float32, device=device)

    n = Xt.shape[0]
    idx = np.arange(n)

    for _ in range(epochs):
        np.random.shuffle(idx)
        model.train()
        for s0 in range(0, n, batch_size):
            b = idx[s0:s0+batch_size]
            xb = Xt[b]
            opt.zero_grad()
            s_code, xhat = model(xb)
            loss = ((xhat - xb)**2).mean() + l1 * s_code.abs().mean()
            loss.backward()
            opt.step()

    return model


# ============================================================
# 5) Geometry cosines (unchanged)
# ============================================================

def fisher_cosine_matrix(W: np.ndarray, D: np.ndarray, Fm: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Fsym = 0.5 * (Fm + Fm.T)
    WFW = np.einsum("ih,hk,ik->i", W, Fsym, W)
    DFD = np.einsum("jh,hk,jk->j", D, Fsym, D)
    Wn = np.sqrt(np.maximum(WFW, eps))
    Dn = np.sqrt(np.maximum(DFD, eps))
    num = W @ Fsym @ D.T
    return num / (Wn[:, None] * Dn[None, :] + eps)


def euclid_cosine_matrix(W: np.ndarray, D: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Wn = np.sqrt(np.sum(W * W, axis=1, keepdims=True) + eps)
    Dn = np.sqrt(np.sum(D * D, axis=1, keepdims=True) + eps)
    return (W @ D.T) / (Wn @ Dn.T)


def fisher_cosine_self(W: np.ndarray, Fm: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Fsym = 0.5 * (Fm + Fm.T)
    WFW = np.einsum("ih,hk,ik->i", W, Fsym, W)
    Wn = np.sqrt(np.maximum(WFW, eps))
    num = W @ Fsym @ W.T
    return num / (Wn[:, None] * Wn[None, :] + eps)


def euclid_cosine_self(W: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    Wn = np.sqrt(np.sum(W * W, axis=1) + eps)
    num = W @ W.T
    return num / (Wn[:, None] * Wn[None, :] + eps)


# ============================================================
# 6) Pair frustration + faithfulness helpers (unchanged)
# ============================================================

def pair_soft_frustration_metric(S: np.ndarray, Z: np.ndarray, *, eps: float = 1e-6) -> dict:
    k, K = S.shape
    if k < 2:
        return {
            "pair_soft_mean": 0.0, "pair_soft_max": 0.0, "pair_soft_nonzero_frac": 0.0,
            "pair_soft_pairs": 0, "pair_soft_eps": float(eps), "mean_abs_Z": 0.0,
            "pair_raw_mean": 0.0, "pair_raw_max": 0.0, "pair_raw_nonzero_frac": 0.0,
        }

    soft_scores, raw_scores = [], []
    soft_nonzero = 0
    raw_nonzero = 0
    total_pairs = 0

    for l in range(k):
        for r in range(l + 1, k):
            total_pairs += 1
            z = float(Z[l, r])
            if z == 0.0:
                soft_scores.append(0.0); raw_scores.append(0.0)
                continue

            prod = S[l, :] * S[r, :]
            zsign = np.sign(z)
            psign = np.sign(prod)

            contr_mask = (psign != 0) & (psign != zsign)
            if not np.any(contr_mask):
                soft_scores.append(0.0); raw_scores.append(0.0)
                continue

            best_raw = float(np.max(np.abs(prod[contr_mask])))
            raw_scores.append(best_raw)
            if best_raw > 0.0:
                raw_nonzero += 1

            best_soft = best_raw / (abs(z) + eps)
            soft_scores.append(best_soft)
            if best_soft > 0.0:
                soft_nonzero += 1

    soft_scores = np.asarray(soft_scores, dtype=float)
    raw_scores = np.asarray(raw_scores, dtype=float)

    tri = np.triu_indices(k, 1)
    mean_abs_Z = float(np.mean(np.abs(Z[tri])))

    return {
        "pair_soft_mean": float(soft_scores.mean()) if soft_scores.size else 0.0,
        "pair_soft_max": float(soft_scores.max()) if soft_scores.size else 0.0,
        "pair_soft_nonzero_frac": float(soft_nonzero / total_pairs) if total_pairs else 0.0,
        "pair_soft_pairs": int(total_pairs),
        "pair_soft_eps": float(eps),
        "mean_abs_Z": float(mean_abs_Z),
        "pair_raw_mean": float(raw_scores.mean()) if raw_scores.size else 0.0,
        "pair_raw_max": float(raw_scores.max()) if raw_scores.size else 0.0,
        "pair_raw_nonzero_frac": float(raw_nonzero / total_pairs) if total_pairs else 0.0,
    }


def frob_norm(A: np.ndarray) -> float:
    return float(np.sqrt(np.sum(A * A)))


def frob_abs_rel(A: np.ndarray, B: np.ndarray, eps: float = 1e-12) -> dict:
    diff = A - B
    abs_f = frob_norm(diff)
    denom = frob_norm(A) + eps
    rel_f = abs_f / denom
    return {"frob_abs": float(abs_f), "frob_rel": float(rel_f)}


def cov_matrix(X: np.ndarray) -> np.ndarray:
    return np.cov(X, rowvar=False, bias=False)


def corr_from_cov(C: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    d = np.sqrt(np.maximum(np.diag(C), eps))
    return C / (d[:, None] * d[None, :] + eps)


# ============================================================
# 6.5) C3 prediction from frustrated vs non-frustrated atoms (unchanged)
# ============================================================

def _select_frustrated_atoms_pair12(S: np.ndarray, Z: np.ndarray, *, tiny: float = 1e-15) -> np.ndarray:
    z = float(Z[0, 1])
    if z == 0.0:
        return np.array([], dtype=np.int64)

    prod = S[0, :] * S[1, :]
    mask_nz = np.abs(prod) > tiny
    zsign = np.sign(z)
    psign = np.sign(prod)

    contr = mask_nz & (psign != 0) & (psign != zsign)
    return np.where(contr)[0].astype(np.int64)


def _project_X_onto_atoms(X: np.ndarray, D: np.ndarray, idx: np.ndarray) -> np.ndarray:
    if idx.size == 0:
        return np.zeros((X.shape[0], 0), dtype=np.float32)
    return (X @ D[idx, :].T).astype(np.float32)


def _ridge_predict(Xtr: np.ndarray, ytr: np.ndarray, Xte: np.ndarray, *, lam: float = 1e-3):
    Xtr = np.asarray(Xtr, dtype=np.float64)
    Xte = np.asarray(Xte, dtype=np.float64)
    ytr = np.asarray(ytr, dtype=np.float64).reshape(-1)

    x_mu = Xtr.mean(axis=0, keepdims=True) if Xtr.shape[1] else np.zeros((1, 0), dtype=np.float64)
    y_mu = float(ytr.mean())

    Xc = Xtr - x_mu
    yc = ytr - y_mu

    d = Xc.shape[1]
    if d == 0:
        yhat = np.full((Xte.shape[0],), y_mu, dtype=np.float32)
        return yhat, (y_mu, np.zeros((0,), dtype=np.float32))

    A = Xc.T @ Xc + lam * np.eye(d)
    b = np.linalg.solve(A, Xc.T @ yc)
    b0 = y_mu - float((x_mu @ b.reshape(-1, 1)).squeeze())
    yhat = (Xte @ b) + b0
    return yhat.astype(np.float32), (float(b0), b.astype(np.float32))


def _mse_r2(y_true: np.ndarray, y_pred: np.ndarray):
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1)
    mse = float(np.mean((y_true - y_pred) ** 2))
    var = float(np.var(y_true))
    r2 = float(1.0 - mse / (var + 1e-12))
    return mse, r2


# ============================================================
# 7) Data build: Sarcasm Y + RoBERTa logits C + DeBERTa embeddings X
# ============================================================

def _subsample_indices(n_total: int, n_keep: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    if n_keep >= n_total:
        return np.arange(n_total, dtype=np.int64)
    return rng.choice(np.arange(n_total, dtype=np.int64), size=n_keep, replace=False)


@torch.no_grad()
def _batched_roberta_sentiment_logits(texts, model_name: str, *, batch_size: int, max_len: int, device):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
    mdl.eval()

    id2label = {int(k): str(v).lower() for k, v in mdl.config.id2label.items()}
    if any(("neg" in v or "negative" in v) for v in id2label.values()):
        def _find(targets):
            for i, lab in id2label.items():
                if any(t in lab for t in targets):
                    return i
            return None
        neg_i = _find(["neg", "negative"])
        neu_i = _find(["neu", "neutral"])
        pos_i = _find(["pos", "positive"])
        if None in (neg_i, neu_i, pos_i):
            neg_i, neu_i, pos_i = 0, 1, 2
    else:
        neg_i, neu_i, pos_i = 0, 1, 2

    all_logits = []
    for i in range(0, len(texts), batch_size):
        batch = [("" if t is None else str(t)) for t in texts[i:i+batch_size]]
        inputs = tok(
            batch, return_tensors="pt",
            truncation=True, padding=True, max_length=max_len
        ).to(device)
        logits = mdl(**inputs).logits
        all_logits.append(logits.detach().cpu())

    L = torch.cat(all_logits, dim=0).numpy().astype(np.float32)
    z_neg = L[:, neg_i]
    z_neu = L[:, neu_i]
    z_pos = L[:, pos_i]
    C = np.stack([z_pos, z_neg, z_neu], axis=1).astype(np.float32)
    return C


@torch.no_grad()
def _batched_deberta_cls_embeddings(texts, model_name: str, *, batch_size: int, max_len: int, device):
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModel.from_pretrained(model_name).to(device)
    mdl.eval()

    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = [("" if t is None else str(t)) for t in texts[i:i+batch_size]]
        inputs = tok(
            batch, return_tensors="pt",
            truncation=True, padding=True, max_length=max_len
        ).to(device)

        out = mdl(**inputs)
        cls = out.last_hidden_state[:, 0, :]
        vecs.append(cls.detach().cpu())

    X = torch.cat(vecs, dim=0).numpy().astype(np.float32)
    return X


def build_or_load_cached_dataset():
    if os.path.exists(CACHE_NPZ):
        print("Loading cached dataset:", CACHE_NPZ)
        z = np.load(CACHE_NPZ, allow_pickle=True)
        X = z["X"].astype(np.float32)
        C = z["C"].astype(np.float32)
        Y = z["Y"].astype(np.int64)
        texts = z["texts"].tolist()
        return X, C, Y, texts

    # ============================================================
    # ### NEW ### Load local sarcasm headlines JSONL files
    # ============================================================
    frames = []
    for fn in FILES:
        path = os.path.join(DATA_DIR, fn)
        if not os.path.exists(path):
            raise FileNotFoundError(f"Missing sarcasm data file: {path}")
        df0 = load_jsonl(path)
        frames.append(df0)

    df = pd.concat(frames, axis=0, ignore_index=True)

    # expected columns: "headline" and "is_sarcastic"
    if "headline" not in df.columns:
        raise ValueError(f"Expected column 'headline' in sarcasm dataset, got columns: {list(df.columns)}")
    if "is_sarcastic" not in df.columns:
        raise ValueError(f"Expected column 'is_sarcastic' in sarcasm dataset, got columns: {list(df.columns)}")

    # light cleaning
    df = df.dropna(subset=["headline", "is_sarcastic"]).copy()
    df["headline"] = df["headline"].astype(str)
    df["is_sarcastic"] = df["is_sarcastic"].astype(int)

    # ensure binary
    df = df[df["is_sarcastic"].isin([0, 1])].reset_index(drop=True)

    print(
        f"[DEBUG] Sarcasm full size={len(df)} | prevalence={df['is_sarcastic'].mean():.4f} "
        f"({df['is_sarcastic'].sum()}/{len(df)})"
    )

    idx = _subsample_indices(len(df), N_DATA, SEED_DATA)
    df = df.iloc[idx].reset_index(drop=True)

    texts = df["headline"].tolist()
    Y = df["is_sarcastic"].to_numpy(dtype=np.int64)

    print(
        f"[DEBUG] Sarcasm subsample size={len(df)} | prevalence={Y.mean():.4f} "
        f"({int(Y.sum())}/{len(Y)})"
    )

    print("Computing RoBERTa sentiment logits (concepts)...")
    C = _batched_roberta_sentiment_logits(
        texts, SENT_MODEL, batch_size=BATCH_SIZE_SENT, max_len=MAX_LEN, device=device
    )

    print("Computing DeBERTa CLS embeddings (activations a)...")
    X = _batched_deberta_cls_embeddings(
        texts, EMB_MODEL, batch_size=BATCH_SIZE_EMB, max_len=MAX_LEN, device=device
    )

    np.savez_compressed(CACHE_NPZ, X=X, C=C, Y=Y, texts=np.array(texts, dtype=object))
    print("Saved cache:", CACHE_NPZ)
    return X, C, Y, texts


# ============================================================
# 8) Stratified folds (unchanged)
# ============================================================

def make_stratified_folds(y: np.ndarray, k_folds: int, seed: int):
    y = np.asarray(y, dtype=np.int64).reshape(-1)
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    rng = np.random.default_rng(seed)
    rng.shuffle(pos); rng.shuffle(neg)

    pos_folds = np.array_split(pos, k_folds)
    neg_folds = np.array_split(neg, k_folds)

    folds = []
    all_idx = np.arange(len(y), dtype=np.int64)
    for k in range(k_folds):
        te = np.concatenate([pos_folds[k], neg_folds[k]])
        te = np.unique(te)
        mask = np.ones(len(y), dtype=bool)
        mask[te] = False
        tr = all_idx[mask]
        folds.append((tr, te))
    return folds


# ============================================================
# 9) One fold run (UNCHANGED from your last version)
# ============================================================

def run_one_fold(
    *,
    fold_id: int,
    seed: int,
    X: np.ndarray,
    C: np.ndarray,
    y: np.ndarray,
    tr_idx: np.ndarray,
    te_idx: np.ndarray,
    device="cpu",
    p_lo=0.2,
    p_hi=0.8,
    min_keep=200,
    K_sae=60,
    pair_soft_eps=1e-6,
    ridge_lam_c3=1e-3,
):
    r = X.shape[1]

    B = np.cov(C, rowvar=False, bias=False).astype(np.float32)

    X_tr, X_te = X[tr_idx].astype(np.float32), X[te_idx].astype(np.float32)
    y_tr, y_te = y[tr_idx], y[te_idx]

    if STANDARDIZE_X_PER_FOLD:
        x_mu = X_tr.mean(axis=0, keepdims=True)
        x_sd = X_tr.std(axis=0, keepdims=True) + NORM_EPS
        X_tr = (X_tr - x_mu) / x_sd
        X_te = (X_te - x_mu) / x_sd

    bb, bb_acc = train_bb_minibatch(
        X_tr, y_tr, X_te, y_te,
        hidden=HIDDEN_BB, epochs=EPOCHS_BB, batch_size=512, lr=1e-3,
        seed=seed, device=device
    )

    with torch.no_grad():
        bb_logits_te = bb(torch.tensor(X_te, dtype=torch.float32, device=device))
        bb_f1 = f1_from_logits(bb_logits_te, y_te, thresh=0.5)

    p_lo_use, p_hi_use = (p_lo, p_hi) if p_lo < p_hi else (p_hi, p_lo)
    p_tr = bb_predict_proba(bb, X_tr, device=device)
    keep = np.where((p_tr >= p_lo_use) & (p_tr <= p_hi_use))[0]
    if keep.size < min_keep:
        order = np.argsort(np.abs(p_tr - 0.5))
        keep = order[:min_keep]
    Fm = compute_fisher_on_input_x(bb, X_tr[keep], device=device, ridge=1e-6)

    sae = train_sae_minibatch(
        X_tr,
        K=K_sae, epochs=EPOCHS_SAE, batch_size=512, lr=2e-3, l1=1e-3,
        seed=seed, device=device
    )
    D = sae.D.detach().cpu().numpy().astype(np.float32)

    C_tr_raw = C[tr_idx].astype(np.float32)
    C_te_raw = C[te_idx].astype(np.float32)

    if NORMALIZE_C_FOR_CBM:
        c_mu = C_tr_raw.mean(axis=0, keepdims=True)
        c_sd = C_tr_raw.std(axis=0, keepdims=True) + NORM_EPS
        C_tr_norm = (C_tr_raw - c_mu) / c_sd
        C_te_norm = (C_te_raw - c_mu) / c_sd
    else:
        C_tr_norm, C_te_norm = C_tr_raw, C_te_raw

    def _run_cbm_block(k_known: int, tag: str):
        Ck_tr = C_tr_norm[:, :k_known].astype(np.float32)
        Ck_te = C_te_norm[:, :k_known].astype(np.float32)
        B_known = np.array(B[:k_known, :k_known], dtype=float)

        cbm, cbm_acc, cbm_mse = train_cbm_linear_minibatch(
            X_tr, Ck_tr, y_tr,
            X_te, Ck_te, y_te,
            epochs=EPOCHS_CBM, batch_size=512, lr=1e-3,
            lambda_c=LAMBDA_C, seed=seed, device=device
        )
        Wc = cbm.concept.weight.detach().cpu().numpy().astype(np.float32)

        with torch.no_grad():
            Xv = torch.tensor(X_te, dtype=torch.float32, device=device)
            c_hat_te, logit_te = cbm(Xv)
            C_hat_te_norm = c_hat_te.detach().cpu().numpy().astype(np.float32)
            cbm_f1 = f1_from_logits(logit_te, y_te, thresh=0.5)

        if NORMALIZE_C_FOR_CBM:
            mu_k = c_mu[:, :k_known]
            sd_k = c_sd[:, :k_known]
            C_hat_te_raw = (C_hat_te_norm * sd_k) + mu_k
        else:
            C_hat_te_raw = C_hat_te_norm

        S_f = fisher_cosine_matrix(Wc, D, Fm)
        S_e = euclid_cosine_matrix(Wc, D)
        Z_f = fisher_cosine_self(Wc, Fm)
        Z_e = euclid_cosine_self(Wc)

        pairF = pair_soft_frustration_metric(S_f, Z_f, eps=PAIR_SOFT_EPS)
        pairE = pair_soft_frustration_metric(S_e, Z_e, eps=PAIR_SOFT_EPS)

        Sd = frob_abs_rel(S_f, S_e)

        Cov_hat = cov_matrix(C_hat_te_raw)
        Corr_hat = corr_from_cov(Cov_hat)
        Corr_B = corr_from_cov(B_known)
        cov_diff = frob_abs_rel(Cov_hat, B_known)
        corr_diff = frob_abs_rel(Corr_hat, Corr_B)

        out = {
            f"{tag}_k_known": int(k_known),
            f"{tag}_acc": float(cbm_acc),
            f"{tag}_f1": float(cbm_f1),
            f"{tag}_mse": float(cbm_mse),

            f"{tag}_F_pair_raw_mean": float(pairF["pair_raw_mean"]),
            f"{tag}_E_pair_raw_mean": float(pairE["pair_raw_mean"]),
            f"{tag}_F_pair_raw_max": float(pairF["pair_raw_max"]),
            f"{tag}_E_pair_raw_max": float(pairE["pair_raw_max"]),

            f"{tag}_S_frob_abs": float(Sd["frob_abs"]),
            f"{tag}_S_frob_rel": float(Sd["frob_rel"]),

            f"{tag}_Cov_frob_abs": float(cov_diff["frob_abs"]),
            f"{tag}_Cov_frob_rel": float(cov_diff["frob_rel"]),
            f"{tag}_Corr_frob_abs": float(corr_diff["frob_abs"]),
            f"{tag}_Corr_frob_rel": float(corr_diff["frob_rel"]),
        }
        return out, (S_f, Z_f, Wc)

    cbm1_metrics, (S1_f, Z1_f, _W1) = _run_cbm_block(2, "cbm1")

    C3_tr = C_tr_raw[:, 2].astype(np.float32)
    C3_te = C_te_raw[:, 2].astype(np.float32)

    frust_idx_full = _select_frustrated_atoms_pair12(S1_f, Z1_f, tiny=1e-15)
    n_frust_full = int(frust_idx_full.size)

    all_idx = np.arange(D.shape[0], dtype=np.int64)
    non_frust_pool = np.setdiff1d(all_idx, frust_idx_full, assume_unique=False)
    n_nonfrust_pool = int(non_frust_pool.size)

    rng_atoms = np.random.default_rng(seed + 99991 + fold_id)
    m_match = int(min(n_frust_full, n_nonfrust_pool))

    if m_match > 0:
        frust_idx = rng_atoms.choice(frust_idx_full, size=m_match, replace=False)
        non_frust_idx = rng_atoms.choice(non_frust_pool, size=m_match, replace=False)
    else:
        frust_idx = np.array([], dtype=np.int64)
        non_frust_idx = np.array([], dtype=np.int64)

    PhiF_tr = _project_X_onto_atoms(X_tr, D, frust_idx)
    PhiF_te = _project_X_onto_atoms(X_te, D, frust_idx)
    PhiN_tr = _project_X_onto_atoms(X_tr, D, non_frust_idx)
    PhiN_te = _project_X_onto_atoms(X_te, D, non_frust_idx)

    yhatF_te, _ = _ridge_predict(PhiF_tr, C3_tr, PhiF_te, lam=ridge_lam_c3)
    yhatN_te, _ = _ridge_predict(PhiN_tr, C3_tr, PhiN_te, lam=ridge_lam_c3)

    C3_mse_frust, C3_r2_frust = _mse_r2(C3_te, yhatF_te)
    C3_mse_nonfrust, C3_r2_nonfrust = _mse_r2(C3_te, yhatN_te)

    ridge_metrics = {
        "n_frust_atoms_full": int(n_frust_full),
        "n_nonfrust_pool": int(n_nonfrust_pool),
        "n_atoms_matched": int(m_match),
        "C3_ridge_lam": float(ridge_lam_c3),
        "C3_mse_frust_atoms": float(C3_mse_frust),
        "C3_r2_frust_atoms": float(C3_r2_frust),
        "C3_mse_nonfrust_atoms": float(C3_mse_nonfrust),
        "C3_r2_nonfrust_atoms": float(C3_r2_nonfrust),
    }

    cbm2_metrics, (S2_f, Z2_f, W2) = _run_cbm_block(3, "cbm2")

    W2_12 = W2[:2, :]
    S2_f_12 = fisher_cosine_matrix(W2_12, D, Fm)
    S2_e_12 = euclid_cosine_matrix(W2_12, D)
    Z2_f_12 = fisher_cosine_self(W2_12, Fm)
    Z2_e_12 = euclid_cosine_self(W2_12)

    pairF2_12 = pair_soft_frustration_metric(S2_f_12, Z2_f_12, eps=PAIR_SOFT_EPS)
    pairE2_12 = pair_soft_frustration_metric(S2_e_12, Z2_e_12, eps=PAIR_SOFT_EPS)

    cbm2_pair12_metrics = {
        "cbm2_F_pair12_raw_mean": float(pairF2_12["pair_raw_mean"]),
        "cbm2_E_pair12_raw_mean": float(pairE2_12["pair_raw_mean"]),
        "cbm2_F_pair12_raw_max":  float(pairF2_12["pair_raw_max"]),
        "cbm2_E_pair12_raw_max":  float(pairE2_12["pair_raw_max"]),
        "cbm2_F_pair12_nonzero_frac": float(pairF2_12["pair_raw_nonzero_frac"]),
        "cbm2_E_pair12_nonzero_frac": float(pairE2_12["pair_raw_nonzero_frac"]),
    }

    row = {
        # ### NEW ### task fields kept but now mean “sarcasm dataset run”
        "task_mode": str(TASK_MODE),
        "task_logic": str(TASK_LOGIC),
        "task_sig": str(TASK_SIG),

        "stdX_per_fold": int(STANDARDIZE_X_PER_FOLD),
        "normC_for_cbm": int(NORMALIZE_C_FOR_CBM),

        "fold": int(fold_id),
        "seed": int(seed),
        "N_total": int(len(X)),
        "N_train": int(len(tr_idx)),
        "N_test": int(len(te_idx)),
        "r": int(r),
        "p_lo": float(p_lo_use),
        "p_hi": float(p_hi_use),
        "F_keep_n": int(len(keep)),
        "bb_acc": float(bb_acc),
        "bb_f1": float(bb_f1),
        "Y_pos_rate_total": float(np.mean(y)),
        "Y_pos_rate_train": float(np.mean(y_tr)),
        "Y_pos_rate_test": float(np.mean(y_te)),
    }
    row.update(cbm1_metrics)
    row.update(ridge_metrics)
    row.update(cbm2_metrics)
    row.update(cbm2_pair12_metrics)
    return row


# ============================================================
# 10) Main (unchanged except filenames already changed)
# ============================================================

def main():
    print(f"\n=== Running SARCASM HEADLINES | N_DATA={N_DATA} ===")
    print(f"    STANDARDIZE_X_PER_FOLD={STANDARDIZE_X_PER_FOLD} | NORMALIZE_C_FOR_CBM={NORMALIZE_C_FOR_CBM}")
    X, C, Y, _texts = build_or_load_cached_dataset()
    assert X.shape[0] == C.shape[0] == Y.shape[0] == N_DATA
    print("Shapes: X", X.shape, "C", C.shape, "Y", Y.shape)

    folds = make_stratified_folds(Y, K_FOLDS, SEED_FOLDS)

    rows = []
    for k, (tr_idx, te_idx) in enumerate(folds):
        print(f"\n=== Fold {k+1}/{K_FOLDS} ===")
        out = run_one_fold(
            fold_id=k,
            seed=SEED_FOLDS + k,
            X=X, C=C, y=Y,
            tr_idx=tr_idx, te_idx=te_idx,
            device=str(device),
            p_lo=P_LO, p_hi=P_HI, min_keep=MIN_KEEP,
            K_sae=K_SAE, pair_soft_eps=PAIR_SOFT_EPS,
            ridge_lam_c3=RIDGE_LAM_C3,
        )
        rows.append(out)

        print(
            f"bb_acc={out['bb_acc']:.3f} bb_f1={out['bb_f1']:.3f} | "
            f"cbm1_acc={out['cbm1_acc']:.3f} cbm1_f1={out['cbm1_f1']:.3f} "
            f"GammaF1={out['cbm1_F_pair_raw_mean']:.4f} | "
            f"cbm2_acc={out['cbm2_acc']:.3f} cbm2_f1={out['cbm2_f1']:.3f} "
            f"GammaF2(allpairs)={out['cbm2_F_pair_raw_mean']:.4f} "
            f"GammaF2(pair12)={out['cbm2_F_pair12_raw_mean']:.4f} | "
            f"nF(full/match)={out['n_frust_atoms_full']}/{out['n_atoms_matched']} "
            f"C3R2(F/N)={out['C3_r2_frust_atoms']:.3f}/{out['C3_r2_nonfrust_atoms']:.3f}"
        )

    df = pd.DataFrame(rows)
    df.to_csv(RESULTS_CSV, index=False)
    print("\nSaved results:", RESULTS_CSV)

    # keep your original describe; if it truncates in your console, we can tweak display options separately
    print(df.describe(include="all").T[["mean","std","min","max"]])


if __name__ == "__main__":
    main()


Using device: cuda

=== Running SARCASM HEADLINES | N_DATA=10000 ===
    STANDARDIZE_X_PER_FOLD=True | NORMALIZE_C_FOR_CBM=True
Loading cached dataset: goemotions_frustration_runs/cache_sarcasm_0d8b78b507_N10000_seed123.npz
Shapes: X (10000, 768) C (10000, 3) Y (10000,)

=== Fold 1/20 ===
bb_acc=0.892 bb_f1=0.882 | cbm1_acc=0.657 cbm1_f1=0.598 GammaF1=0.0912 | cbm2_acc=0.756 cbm2_f1=0.730 GammaF2(allpairs)=0.0009 GammaF2(pair12)=0.0019 | nF(full/match)=24/24 C3R2(F/N)=0.146/0.098

=== Fold 2/20 ===
bb_acc=0.932 bb_f1=0.926 | cbm1_acc=0.723 cbm1_f1=0.599 GammaF1=0.0116 | cbm2_acc=0.866 cbm2_f1=0.852 GammaF2(allpairs)=0.0009 GammaF2(pair12)=0.0023 | nF(full/match)=5/5 C3R2(F/N)=0.067/0.004

=== Fold 3/20 ===
bb_acc=0.894 bb_f1=0.885 | cbm1_acc=0.758 cbm1_f1=0.734 GammaF1=0.0090 | cbm2_acc=0.758 cbm2_f1=0.746 GammaF2(allpairs)=0.0009 GammaF2(pair12)=0.0007 | nF(full/match)=8/8 C3R2(F/N)=0.058/0.087

=== Fold 4/20 ===
bb_acc=0.928 bb_f1=0.921 | cbm1_acc=0.814 cbm1_f1=0.814 GammaF1=0.0072 |